# Full Test Set Visualization (PDF & TIFF Export)

This notebook:
1. Re-creates the **Test Set** split.
2. Iterates through all test samples.
3. Generates a **2x4 grid** (Ground Truth + Unfiltered + Filtered Regions).
4. **Saves outputs as both PDF and TIFF formats.**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy.ma as ma
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.auto import tqdm


C:\Users\user_picm\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Configuration

In [2]:
class Config:
    # === PATHS (Update this) ===
    DATA_DIR = Path(r"F:\MPL_Data\mmNoTissueFilter\TRIMMM")
    OUTPUT_DIR = Path("test_set_viz_pdf_tiff")

    # === SPLIT SETTINGS (Must match training) ===
    ISOLATED_SAMPLES = [
        'Day6_mm_results_Day6E_4B_S2',
        'Day6_mm_results_day6_3',
        'Day0_mm_results_Day0G_7B_S3',
        'Day0_mm_results_Day0H_3B_S6'
    ]
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    RANDOM_SEED = 42

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# 2. Data Discovery & Test Set Identification

In [3]:
def get_test_set(data_dir):
    """Replicates the dataset splitting logic to find the Test Set."""
    if not data_dir.exists():
        raise FileNotFoundError(f"Directory not found: {data_dir}")

    # 1. Discovery
    samples = []
    parent_dirs = [d for d in data_dir.iterdir() if d.is_dir() and not d.name.startswith('.')]

    for parent_dir in parent_dirs:
        group_id = parent_dir.name
        # Group variations
        sample_groups = {}
        for sample_dir in [d for d in parent_dir.iterdir() if d.is_dir()]:
            base_name = sample_dir.name
            for suffix in ['_original', '_rot90', '_rot180', '_rot270', '_flip_h', '_flip_v']:
                base_name = base_name.replace(suffix, '')

            unique_name = f"{parent_dir.name}_{base_name}"
            if unique_name not in sample_groups: sample_groups[unique_name] = []
            sample_groups[unique_name].append(sample_dir)

        for unique_name, dirs in sample_groups.items():
            selected_dir = next((d for d in dirs if '_original' in d.name), dirs[0])
            npz_files = [f for f in selected_dir.glob("*.npz") if not f.name.startswith('.')]
            if npz_files:
                samples.append({'sample_name': unique_name, 'group_id': group_id, 'npz_path': npz_files[0]})

    # 2. Remove Isolated
    dataset_samples = [s for s in samples if s['sample_name'] not in Config.ISOLATED_SAMPLES]

    # 3. Stratified Split
    groups = [s['group_id'] for s in dataset_samples]
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=Config.VAL_RATIO + Config.TEST_RATIO, random_state=Config.RANDOM_SEED)
    _, temp_idx = next(splitter.split(dataset_samples, groups))
    temp_subset = [dataset_samples[i] for i in temp_idx]

    temp_groups = [temp_subset[i]['group_id'] for i in range(len(temp_subset))]
    splitter_val = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=Config.RANDOM_SEED)
    _, test_idx = next(splitter_val.split(temp_subset, temp_groups))

    return [temp_subset[i] for i in test_idx]

test_samples = get_test_set(Config.DATA_DIR)
print(f"Identified {len(test_samples)} samples in the Test Set.")


Identified 11 samples in the Test Set.


# 3. Data Loading Helper

In [4]:
def load_data(npz_path):
    try:
        with np.load(npz_path, allow_pickle=True) as data:
            # M11
            if 'nM11s' in data: m11 = np.array(data['nM11s'])
            elif 'M11s' in data:
                raw = np.array(data['M11s'])
                m11 = (raw - raw.min()) / (raw.max() - raw.min()) if raw.max() > raw.min() else raw
            else: return None, None

            # Masks
            tissue = np.array(data['tissue_mask']) > 0 if 'tissue_mask' in data else None
            if tissue is None:
                if 'annotation_mask' in data: tissue = np.array(data['annotation_mask']) > 0
                else: return None, None

            os = np.array(data['os_mask']) > 0 if 'os_mask' in data else None
            vag = np.array(data['vaginal_mask']) > 0 if 'vaginal_mask' in data else None

            # Build GT (0=Bg, 1=Tissue, 2=OS, 3=Vaginal)
            gt = np.zeros_like(tissue, dtype=np.int64)
            gt[tissue] = 1
            if os is not None: gt[os] = 2
            if vag is not None: gt[vag] = 3

            return m11, gt
    except:
        return None, None


# 4. Visualization Engine (Saves PDF & TIFF)

**Row 1:** Ground Truth visualizations
**Row 2:** Unfiltered M11 + All filtered versions (Inclusive Tissue, OS, Vaginal)

In [5]:
def visualize_and_save(sample, output_dir):
    m11, gt_mask = load_data(sample['npz_path'])
    if m11 is None: return

    # --- SETUP COLORS ---
    # 0=Black, 1=Blue (Tissue), 2=Green (OS), 3=Red (Vaginal)
    cmap_colors = ['black', 'blue', 'lime', 'red']
    cmap = mcolors.ListedColormap(cmap_colors)
    norm = mcolors.BoundaryNorm([0, 1, 2, 3, 4], cmap.N)

    # --- DEFINE MASKS ---
    mask_os = (gt_mask == 2)
    mask_vag = (gt_mask == 3)

    # *** CUSTOM LOGIC: Inclusive Tissue ***
    # Tissue Filter = (Tissue OR OS OR Vaginal)
    mask_inclusive_tissue = (gt_mask > 0)

    # --- CREATE PLOT ---
    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    safe_name = sample['sample_name'].replace("_mm_results", "")
    plt.suptitle(f"Sample: {safe_name}", fontsize=16)

    # === ROW 1: GROUND TRUTH & COLORS ===

    # 1. Original M11
    axes[0, 0].imshow(m11, cmap='gray')
    axes[0, 0].set_title("Original M11")
    axes[0, 0].axis('off')

    # 2. Colored Mask Map
    axes[0, 1].imshow(gt_mask, cmap=cmap, norm=norm, interpolation='nearest')
    axes[0, 1].set_title("Ground Truth Map\n(Blue=T, Green=OS, Red=Vag)")
    axes[0, 1].axis('off')

    # 3. Overlay
    axes[0, 2].imshow(m11, cmap='gray')
    axes[0, 2].imshow(gt_mask, cmap=cmap, norm=norm, alpha=0.4, interpolation='nearest')
    axes[0, 2].set_title("Overlay on M11")
    axes[0, 2].axis('off')

    # 4. Empty placeholder for symmetry
    axes[0, 3].axis('off')

    # === ROW 2: FILTERED REGIONS ===

    # 5. Completely Unfiltered M11
    axes[1, 0].imshow(m11, cmap='gray')
    axes[1, 0].set_title("M11: Unfiltered")
    axes[1, 0].axis('off')

    # 6. Inclusive Tissue Filter (Tissue + OS + Vaginal)
    if mask_inclusive_tissue.any():
        masked_tissue = ma.array(m11, mask=~mask_inclusive_tissue)
        axes[1, 1].imshow(masked_tissue, cmap='gray')
        axes[1, 1].set_title("M11: Inclusive Tissue\n(Tissue + OS + Vaginal)")
    else:
        axes[1, 1].text(0.5, 0.5, "Empty", ha='center')
    axes[1, 1].axis('off')

    # 7. OS Filter
    if mask_os.any():
        masked_os = ma.array(m11, mask=~mask_os)
        axes[1, 2].imshow(masked_os, cmap='gray')
        axes[1, 2].set_title("M11: OS Region Only")
    else:
        axes[1, 2].text(0.5, 0.5, "No OS", ha='center')
    axes[1, 2].axis('off')

    # 8. Vaginal Filter
    if mask_vag.any():
        masked_vag = ma.array(m11, mask=~mask_vag)
        axes[1, 3].imshow(masked_vag, cmap='gray')
        axes[1, 3].set_title("M11: Vaginal Region Only")
    else:
        axes[1, 3].text(0.5, 0.5, "No Vaginal", ha='center')
    axes[1, 3].axis('off')

    plt.tight_layout()

    # --- SAVE COMMANDS ---
    # Save as PDF
    plt.savefig(output_dir / f"{safe_name}_full_viz.pdf", dpi=150)
    # Save as TIFF
    plt.savefig(output_dir / f"{safe_name}_full_viz.tiff", dpi=150)

    plt.close(fig)

# 5. Run

In [6]:
print(f"Saving visualizations to: {Config.OUTPUT_DIR.resolve()}")
for sample in tqdm(test_samples, desc="Visualizing Test Set"):
    visualize_and_save(sample, Config.OUTPUT_DIR)

print("Done.")

Saving visualizations to: C:\Users\user_picm\Desktop\mmTissueFilter\archive\test_set_viz_pdf_tiff


Visualizing Test Set: 100%|██████████| 11/11 [01:04<00:00,  5.85s/it]

Done.
